In [1]:
# from google.colab import drive
# drive.mount("/gdrive")
# current_dir = "/gdrive/MyDrive/Challange1"
# %cd $current_dir
# %pip install optuna

## 1. Hyperparameters

In [2]:
###########
# Model parameters
###########
RNN_TYPE = 'GRU'            # 'RNN', 'LSTM', or 'GRU'
BIDIRECTIONAL = True        # True / False
################
# DATA LOADER PARAMETERS
################
BATCH_SIZE = 512
################
# TRAINING PARAMETERS
################
LEARNING_RATE = 0.001
EPOCHS = 30
PATIENCE = 10
################
# ARCHITECTURE PARAMETERS
################
HIDDEN_LAYERS = 2        # Hidden layers
HIDDEN_SIZE = 64          #42  # Neurons per layer -> prev hidden size = 128
DROPOUT_RATE = 0.2      # Dropout probability
L1_LAMBDA = 1e-7        # L1 penalty
L2_LAMBDA = 1e-2          # L2 penalty for the optimizer
################
# CONV LAYER PARAMETERS
################
CONV_CHANNELS = [64]
CONV_KERNEL_SIZES = [3]
CONV_DROPOUT_RATE = 0.6
##########
# Split parameters
##########
N_VAL_USERS = 120
N_TEST_USERS = 120
################
# Sequence parameters
################
WINDOW_SIZE = 10
STRIDE = 2
################
# CONSTANTS
################
NUM_CLASS = 3
################
# GRID SEARCH / CV
################
USE_KFOLD = False   # True: use K-Fold CV in Optuna; False: single split
SINGLE_VAL_FRACTION = 0.2  # fraction of data used for validation when not using K-Fold

GRID_SEED = 42
N_TRIALS = 2         # adapt as needed
MAX_EPOCHS = 60            # max epochs per trial
GRID_PATIENCE = 10         # early stopping patience per trial
N_FOLDS = 2                # K-fold CV (used only if USE_KFOLD=True)
VERBOSE_GRID = 10

SAVE_DIR = "optuna_top20"

## 2. Import Libraries

### 2.1 Setup Filenames

In [3]:
from datetime import datetime

# Get current date and time for submission filename
current_datetime = datetime.now().strftime("%d-%m-%H-%M")

if BIDIRECTIONAL:
    EXPERIMENT_NAME = f"{RNN_TYPE}_bi_{current_datetime}"
else:
    EXPERIMENT_NAME = f"{RNN_TYPE}_{current_datetime}"

SUBMISSION_FILENAME = f"{EXPERIMENT_NAME}.csv"
print(f"Experiment name: {EXPERIMENT_NAME}")
print(f"Submission filename: {SUBMISSION_FILENAME}")

Experiment name: GRU_bi_17-11-21-28
Submission filename: GRU_bi_17-11-21-28.csv


### 2.2 Import the Libraries

In [4]:
# Set seed for reproducibility
SEED = 42

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Directory configuration
logs_dir = "tensorboard"
models_dir = "models"

models_dir = "models/"

# Model save/load paths
MODEL_SAVE_PATH = f"{models_dir}/{EXPERIMENT_NAME}_model.pt"
MODEL_LOAD_PATH = f"{models_dir}/{EXPERIMENT_NAME}_model.pt"

!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p {models_dir}

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import copy
import shutil
from datetime import datetime
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import torch.nn.functional as F
import torch.optim as optim

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

'pkill' is not recognized as an internal or external command,
operable program or batch file.
The syntax of the command is incorrect.


PyTorch version: 2.5.1+cu121
Device: cuda


## 3. Load Data (Training and Submission)

In [5]:
import pandas as pd

X_train = pd.read_csv('pirate_pain_train.csv')
X_test = pd.read_csv('pirate_pain_test.csv')

y_train = pd.read_csv('pirate_pain_train_labels.csv')

## 4. Dataset Preprocessing

In [6]:
# Set Number of Pirates
NUM_PIRATES = X_train['sample_index'].nunique()

# Merge features and labels
data = X_train.merge(y_train, on='sample_index')

cols = ['n_legs', 'n_hands', 'n_eyes']
unique_values = {col: X_train[col].unique().tolist() for col in cols}


In [7]:
# Create binary features (1 = has prosthetic, 0 = does not)
df_corr_check = data.copy()
df_corr_check['has_peg_leg'] = np.where(df_corr_check['n_legs'] == 'one+peg_leg', 1, 0)
df_corr_check['has_hook_hand'] = np.where(df_corr_check['n_hands'] == 'one+hook_hand', 1, 0)
df_corr_check['has_eye_patch'] = np.where(df_corr_check['n_eyes'] == 'one+eye_patch', 1, 0)

# Map the Label to numeric for the correlation matrix
label_mapping = {'no_pain': 0, 'low_pain': 1, 'high_pain': 2}
df_corr_check['label'] = df_corr_check['label'].map(label_mapping)

# Drop the old features for n_legs, n_hands, n_eyes
data = df_corr_check.copy()
data = data.drop(columns=['n_legs', 'n_hands', 'n_eyes'])


In [8]:
# Drop joint_30
list_to_remove = ['joint_30']

if data.columns.isin(list_to_remove).any():
  data = data.drop(columns=list_to_remove)
  data.head()
else:
  print("Useless features already removed")

In [9]:
data.head()

,sample_index,time,pain_survey_1,pain_survey_2,pain_survey_3,pain_survey_4,joint_00,joint_01,joint_02,joint_03,...,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29,label,has_peg_leg,has_hook_hand,has_eye_patch
0,0,0,2,0,2,1,1.094705,0.985281,1.018302,1.010385,...,1.153299e-05,0.000004,0.017592,0.013508,0.026798,0.027815,0,0,0,0
1,0,1,2,2,2,2,1.135183,1.021175,0.994343,1.052364,...,4.643774e-08,0.000000,0.013352,0.000000,0.013377,0.013716,0,0,0,0
2,0,2,2,0,2,2,1.080745,0.962842,1.009588,0.977169,...,2.424536e-06,0.000003,0.016225,0.008110,0.024097,0.023105,0,0,0,0
3,0,3,2,2,2,2,0.938017,1.081592,0.998021,0.987283,...,5.432416e-08,0.000000,0.011832,0.007450,0.028613,0.024648,0,0,0,0
4,0,4,2,2,2,2,1.090185,1.032145,1.008710,0.963658,...,5.825366e-08,0.000007,0.005360,0.002532,0.033026,0.025328,0,0,0,0


In [10]:
# Count the continouse and categorical features
continuous_cols = []
categorical_cols = []

for col in data.columns:
    if col != 'sample_index' and not col.startswith('pain_survey_') and not col.startswith('has_') and not col == 'label' and not col == 'time':
        continuous_cols.append(col)
    elif col.startswith('has_') or col.startswith('pain_survey_'):
        categorical_cols.append(col)

print(f"Continuous features: {len(continuous_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

Continuous features: 30
Categorical features: 7


In [11]:

# Convert all columns to float32 except 'label', which stays int
label = data['label'].astype(np.int64)
data = data.astype(np.float32)
data['label'] = label


In [12]:
# Check the columns and their types
print("Columns in data:")
print(data.columns.tolist())
print("\nData types:")
print(data.dtypes)

Columns in data:
['sample_index', 'time', 'pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4', 'joint_00', 'joint_01', 'joint_02', 'joint_03', 'joint_04', 'joint_05', 'joint_06', 'joint_07', 'joint_08', 'joint_09', 'joint_10', 'joint_11', 'joint_12', 'joint_13', 'joint_14', 'joint_15', 'joint_16', 'joint_17', 'joint_18', 'joint_19', 'joint_20', 'joint_21', 'joint_22', 'joint_23', 'joint_24', 'joint_25', 'joint_26', 'joint_27', 'joint_28', 'joint_29', 'label', 'has_peg_leg', 'has_hook_hand', 'has_eye_patch']

Data types:
sample_index     float32
time             float32
pain_survey_1    float32
pain_survey_2    float32
pain_survey_3    float32
pain_survey_4    float32
joint_00         float32
joint_01         float32
joint_02         float32
joint_03         float32
joint_04         float32
joint_05         float32
joint_06         float32
joint_07         float32
joint_08         float32
joint_09         float32
joint_10         float32
joint_11         float32
joint_12 

## 5. Train/Val/Test Split

In [13]:


# --- Step 1: Compute each user's dominant label (or label distribution)
user_labels = (
    data.groupby('sample_index')['label']
    .agg(lambda x: x.value_counts().index[0])  # dominant label per user
    .reset_index()
)

train_users, temp_users = train_test_split(
    user_labels['sample_index'],
    test_size=(N_VAL_USERS + N_TEST_USERS) / len(user_labels),
    stratify=user_labels['label'],
    random_state=SEED
)

# Split temp into val/test (also stratified)
temp_labels = user_labels[user_labels['sample_index'].isin(temp_users)]
if N_TEST_USERS != 0:
  val_users, test_users = train_test_split(
      temp_labels['sample_index'],
      test_size=N_TEST_USERS / (N_VAL_USERS + N_TEST_USERS),
      stratify=temp_labels['label'],
      random_state=SEED
  )
else:
  val_users = temp_users
  test_users = []

# --- Step 3: Filter your main df
df_train = data[data['sample_index'].isin(train_users)]
df_val = data[data['sample_index'].isin(val_users)]
df_test = data[data['sample_index'].isin(test_users)]

# --- Step 4: Check label proportions
print("Label proportions:")
print("Train:\n", df_train['label'].value_counts(normalize=True))
print("Val:\n", df_val['label'].value_counts(normalize=True))
print("Test:\n", df_test['label'].value_counts(normalize=True))

Label proportions:
Train:
 label
0    0.771971
1    0.142518
2    0.085511
Name: proportion, dtype: float64
Val:
 label
0    0.775000
1    0.141667
2    0.083333
Name: proportion, dtype: float64
Test:
 label
0    0.775000
1    0.141667
2    0.083333
Name: proportion, dtype: float64


In [14]:

# Define the columns to be normalised

scale_columns = [
    col for col in data.columns
    if (col.startswith('joint_') )
]

scaler = StandardScaler()

# ---- Fit on TRAIN, transform TRAIN / VAL / TEST ----
# Work on copies to keep original df_train, df_val, df_test unchanged if needed
df_train_scaled = df_train.copy()
df_val_scaled = df_val.copy()
df_test_scaled = df_test.copy()

# Fit scaler on train
scaler.fit(df_train_scaled[scale_columns])

# Transform
df_train_scaled[scale_columns] = scaler.transform(df_train_scaled[scale_columns])
df_val_scaled[scale_columns] = scaler.transform(df_val_scaled[scale_columns])
if not df_test_scaled.empty:
    df_test_scaled[scale_columns] = scaler.transform(df_test_scaled[scale_columns])

print("Scaling done.")
print("Train shape:", df_train_scaled.shape)
print("Val shape:", df_val_scaled.shape)
print("Test shape:", df_test_scaled.shape)
# ...existing code...

Scaling done.
Train shape: (67360, 40)
Val shape: (19200, 40)
Test shape: (19200, 40)


## 6. Build Sequences

### 6.1 Build Sequences Function Definition

In [15]:
def build_sequences_enhanced(df, window=200, stride=200, categorical_features=['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4']):
    """
    Enhanced version of build_sequences that separates categorical and continuous features.

    Returns:
    - continuous_data: numpy array of continuous features
    - categorical_data: dict of categorical features
    - labels: numpy array of labels
    """
    # Sanity check to ensure the window is divisible by the stride
    assert window % stride == 0

    # Separate categorical and continuous features
    exclude_cols = ['sample_index', 'label', 'time', 'pirate_id'] + categorical_features
    continuous_cols = [col for col in df.columns if col not in exclude_cols]


    # Initialize lists to store sequences and their corresponding labels
    continuous_dataset = []
    categorical_datasets = {feature: [] for feature in categorical_features}
    labels = []

    # Iterate over unique IDs in the DataFrame
    for id in df['sample_index'].unique():
        # Extract data for the current sample index
        pirate_data = df[df['sample_index'] == id]

        # Extract continuous features
        continuous_temp = pirate_data[continuous_cols].values

        # Extract categorical features
        categorical_temps = {}
        for feature in categorical_features:
            categorical_temps[feature] = pirate_data[feature].values

        # Retrieve the label for the current pirate
        label = pirate_data['label'].values[0]

        # Calculate padding length to ensure full windows
        padding_len = window - len(continuous_temp) % window

        # Create zero padding for continuous features
        continuous_padding = np.zeros((padding_len, len(continuous_cols)), dtype='float32')
        continuous_temp = np.concatenate((continuous_temp, continuous_padding))

        # Create padding for categorical features (use 0 as padding)
        for feature in categorical_features:
            categorical_padding = np.zeros(padding_len, dtype='int64')
            categorical_temps[feature] = np.concatenate((categorical_temps[feature], categorical_padding))

        # Build feature windows and associate them with labels
        idx = 0
        while idx + window <= len(continuous_temp):
            # Continuous features
            continuous_dataset.append(continuous_temp[idx:idx + window])

            # Categorical features
            for feature in categorical_features:
                categorical_datasets[feature].append(categorical_temps[feature][idx:idx + window])

            labels.append(label)
            idx += stride

    # Convert lists to numpy arrays for further processing
    continuous_dataset = np.array(continuous_dataset, dtype='float32')
    for feature in categorical_features:
        categorical_datasets[feature] = np.array(categorical_datasets[feature], dtype='int64')
    labels = np.array(labels)


    return continuous_dataset, categorical_datasets, labels

### 6.2 Build the Sequences

In [16]:
# Generate enhanced sequences for training, validation, and test sets


# Define all categorical features including pirate characteristics
all_categorical_features = ['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4',
                             'has_peg_leg', 'has_hook_hand', 'has_eye_patch']

# Build sequences with separated categorical and continuous features
X_train_cont, X_train_cat, y_train = build_sequences_enhanced(df_train_scaled, WINDOW_SIZE, STRIDE, all_categorical_features)
X_val_cont, X_val_cat, y_val = build_sequences_enhanced(df_val_scaled, WINDOW_SIZE, STRIDE, all_categorical_features)
X_test_cont,  X_test_cat,  y_test_enh = build_sequences_enhanced(df_test_scaled, WINDOW_SIZE, STRIDE, all_categorical_features)

### 6.3 Dataset Definition

In [17]:
# EnhancedDataset updated to include pirate_id (optional)
class EnhancedDataset(torch.utils.data.Dataset):
    def __init__(self, continuous_data, categorical_data, labels, pirate_ids=None):
        """
        continuous_data: np array (N, seq_len, cont_feat)
        categorical_data: dict of feature -> np array (N, seq_len)
        labels: np array (N,)
        pirate_ids: np array (N,) or None  -- integer id per sequence
        """
        self.continuous_data = torch.FloatTensor(continuous_data)
        self.categorical_data = {}
        for feature, data in categorical_data.items():
            self.categorical_data[feature] = torch.LongTensor(data)
        self.labels = torch.LongTensor(labels)
        self.pirate_ids = torch.LongTensor(pirate_ids) if pirate_ids is not None else None

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        cont = self.continuous_data[idx]
        cats = {k: v[idx] for k, v in self.categorical_data.items()}
        label = self.labels[idx]
        if self.pirate_ids is None:
            return cont, cats, label
        else:
            return cont, cats, label, self.pirate_ids[idx]

### 6.4 Data Loaders Definition

In [18]:
def make_enhanced_loader(ds, batch_size, shuffle, drop_last, sampler=None):
    """
    Enhanced data loader that properly handles pirate IDs
    """
    def collate_enhanced_with_ids(batch):
        """Custom collate function that handles pirate IDs"""
        continuous_batch = torch.stack([item[0] for item in batch])

        categorical_batch = {}
        if batch:
            feature_names = batch[0][1].keys()
            for feature in feature_names:
                categorical_batch[feature] = torch.stack([item[1][feature] for item in batch])

        labels_batch = torch.stack([item[2] for item in batch])

        # Check if pirate IDs are present
        if len(batch[0]) == 4:
            pirate_ids_batch = torch.stack([item[3] for item in batch])
            return continuous_batch, categorical_batch, labels_batch, pirate_ids_batch
        else:
            return continuous_batch, categorical_batch, labels_batch

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        drop_last=drop_last,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False,
        collate_fn=collate_enhanced_with_ids,
        sampler=sampler
    )

### 6.5 Create the Datasets

In [19]:

pirate_ids_train = df_train['sample_index'].values
pirate_ids_val = df_val['sample_index'].values
pirate_ids_test = df_test['sample_index'].values


train_enhanced_ds = EnhancedDataset(X_train_cont, X_train_cat, y_train, pirate_ids=pirate_ids_train)
val_enhanced_ds   = EnhancedDataset(X_val_cont,   X_val_cat,   y_val,   pirate_ids=pirate_ids_val)
test_enhanced_ds  = EnhancedDataset(X_test_cont,  X_test_cat,  y_test_enh,  pirate_ids=pirate_ids_test)

print(f"Enhanced dataset sizes:")
print(f"Train: {len(train_enhanced_ds)} samples")
print(f"Val: {len(val_enhanced_ds)} samples")
print(f"Test: {len(test_enhanced_ds)} samples")

Enhanced dataset sizes:
Train: 34101 samples
Val: 9720 samples
Test: 9720 samples


### 6.6 Create the Loaders

In [20]:
# Create enhanced data loaders
train_enhanced_loader = make_enhanced_loader(
                        train_enhanced_ds,
                        batch_size=BATCH_SIZE,
                        shuffle=True,
                        drop_last=False,
                        sampler=None
                    )

val_enhanced_loader = make_enhanced_loader(
                        val_enhanced_ds,
                        batch_size=BATCH_SIZE,
                        shuffle=False,
                        drop_last=False,
                        sampler=None
                    )

test_enhanced_loader = make_enhanced_loader(
                        test_enhanced_ds,
                        batch_size=BATCH_SIZE,
                        shuffle=False,
                        drop_last=False,
                        sampler=None
                    )



## 7. Training and Model Class Definitions

### 7.1 Training Function Definition

In [21]:
def train_enhanced_model(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        scheduler=None,
        epochs=50,
        l1_lambda=None,
        l2_lambda=None,
        patience=15,
        min_delta=0.001,
        scheduler_metric='f1'
        ):
    """
    Train enhanced model with categorical embeddings, regularization, and early stopping.

    Args:
        model: The PyTorch model to train
        train_loader: DataLoader for training data (enhanced format)
        val_loader: DataLoader for validation data (enhanced format)
        criterion: Loss function
        optimizer: Optimizer
        scheduler: Learning rate scheduler (optional)
        epochs: Maximum number of training epochs
        l1_lambda: L1 regularization coefficient (if None, uses L1_LAMBDA global)
        l2_lambda: L2 regularization coefficient (if None, uses L2_LAMBDA global)
        patience: Early stopping patience (epochs to wait without improvement)
        min_delta: Minimum change to qualify as improvement
        scheduler_metric: Metric to use for scheduler ('f1', 'val_loss', or 'combined')

    Returns:
        train_losses, val_losses, val_f1_scores, train_f1_scores, best_epoch: Training history and best epoch
    """
    model.train()
    train_losses = []
    val_losses = []
    val_f1_scores = []
    train_f1_scores = []  # Track training F1 scores for plotting

    # Early stopping variables - track both F1 and validation loss
    best_val_f1 = 0
    best_val_loss = float('inf')
    best_epoch = 0
    best_epoch_loss = 0
    epochs_without_improvement = 0
    epochs_without_loss_improvement = 0
    best_model_state = None

    # Set regularization coefficients
    if l1_lambda is None:
        l1_lambda = L1_LAMBDA if 'L1_LAMBDA' in globals() else 0
    if l2_lambda is None:
        l2_lambda = L2_LAMBDA if 'L2_LAMBDA' in globals() else 0

    print("Starting enhanced model training with Early Stopping...")
    print(f" Regularization: L1={l1_lambda:.2e}, L2={l2_lambda:.2e}")
    print(f" Early Stopping: Patience={patience} epochs, Min improvement={min_delta:.4f}")
    print(f" Scheduler Metric: {scheduler_metric}")
    if scheduler:
        print(f" Using ReduceLROnPlateau scheduler - Mode: {scheduler.mode}, Factor: {scheduler.factor}, Patience: {scheduler.patience}")

    for epoch in range(epochs):
        # Training phase
        model.train()
        epoch_train_loss = 0.0
        num_batches = 0
        # Track training predictions for F1 calculation
        train_predictions = []
        train_true_labels = []

        for batch_data in train_loader:
            # Handle different batch formats (with or without pirate_ids)
            if len(batch_data) == 4:
                continuous_batch, categorical_batch, labels_batch, pirate_ids = batch_data
            else:
                continuous_batch, categorical_batch, labels_batch = batch_data

            # Move to device
            continuous_batch = continuous_batch.to(device)
            categorical_batch = {k: v.to(device) for k, v in categorical_batch.items()}
            labels_batch = labels_batch.to(device)

            # Forward pass (don't pass pirate_ids to model)
            outputs, _ = model(continuous_batch, categorical_batch)
            loss = criterion(outputs, labels_batch)

            # Add L1 and L2 regularization penalties
            if l1_lambda > 0 or l2_lambda > 0:
                l1_norm = 0
                l2_norm = 0

                for param in model.parameters():
                    if l1_lambda > 0:
                        l1_norm += torch.norm(param, 1)
                    if l2_lambda > 0:
                        l2_norm += torch.norm(param, 2) ** 2

                # Add regularization terms to loss
                regularization_loss = l1_lambda * l1_norm + l2_lambda * l2_norm
                loss = loss + regularization_loss

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # Collect predictions for F1 calculation
            with torch.no_grad():
                _, predicted = torch.max(outputs.data, 1)
                train_predictions.extend(predicted.cpu().numpy())
                train_true_labels.extend(labels_batch.cpu().numpy())

            epoch_train_loss += loss.item()
            num_batches += 1


        avg_train_loss = epoch_train_loss / num_batches

        # Calculate training F1 score
        train_f1 = f1_score(train_true_labels, train_predictions, average='weighted')

        # Validation phase
        model.eval()
        val_predictions = []
        val_true_labels = []
        epoch_val_loss = 0.0
        val_batches = 0

        with torch.no_grad():
            for batch_data in val_loader:
                # Handle different batch formats (with or without pirate_ids)
                if len(batch_data) == 4:
                    continuous_batch, categorical_batch, labels_batch, pirate_ids = batch_data
                else:
                    continuous_batch, categorical_batch, labels_batch = batch_data

                # Move to device
                continuous_batch = continuous_batch.to(device)
                categorical_batch = {k: v.to(device) for k, v in categorical_batch.items()}
                labels_batch = labels_batch.to(device)

                # Forward pass (don't pass pirate_ids to model)
                outputs, _ = model(continuous_batch, categorical_batch)
                loss = criterion(outputs, labels_batch)

                # Get predictions
                _, predicted = torch.max(outputs.data, 1)
                val_predictions.extend(predicted.cpu().numpy())
                val_true_labels.extend(labels_batch.cpu().numpy())

                epoch_val_loss += loss.item()
                val_batches += 1

        avg_val_loss = epoch_val_loss / val_batches

        # Calculate validation F1 score
        val_f1 = f1_score(val_true_labels, val_predictions, average='weighted')

        # Store metrics
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        val_f1_scores.append(val_f1)
        train_f1_scores.append(train_f1)

        # Determine the monitoring metric for early stopping
        if scheduler_metric == 'f1':
            current_metric = val_f1
            is_better = current_metric > best_val_f1 + min_delta
        elif scheduler_metric == 'val_loss':
            current_metric = avg_val_loss
            is_better = current_metric < best_val_loss - min_delta
        else:  # combined
            # Normalize and combine both metrics (higher is better)
            normalized_f1 = val_f1
            normalized_loss = 1.0 / (1.0 + avg_val_loss)
            current_metric = (normalized_f1 + normalized_loss) / 2
            best_combined = (best_val_f1 + 1.0 / (1.0 + best_val_loss)) / 2
            is_better = current_metric > best_combined + min_delta

        # Print epoch summary with correct formatting
        print(f"Epoch {epoch+1}/{epochs}: Train Loss: {avg_train_loss:.4f}| Val Loss: {avg_val_loss:.4f}| Train F1: {train_f1:.4f}| Val F1: {val_f1:.4f}")

        # Check for improvement and update best metrics
        if is_better:
            if scheduler_metric == 'f1':
                best_val_f1 = current_metric
            elif scheduler_metric == 'val_loss':
                best_val_loss = current_metric
            else:
                best_val_f1 = val_f1
                best_val_loss = avg_val_loss

            best_epoch = epoch + 1
            epochs_without_improvement = 0
            best_model_state = model.state_dict().copy()
        else:
            epochs_without_improvement += 1

        # Track best validation loss separately
        if avg_val_loss < best_val_loss - min_delta:
            best_val_loss = avg_val_loss
            best_epoch_loss = epoch + 1
            epochs_without_loss_improvement = 0
        else:
            epochs_without_loss_improvement += 1

        # Learning rate scheduling
        if scheduler:
            if scheduler_metric == 'f1':
                scheduler.step(val_f1)
            elif scheduler_metric == 'val_loss':
                scheduler.step(avg_val_loss)
            else:  # combined
                scheduler.step(current_metric)

            # Get current learning rate
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Current LR: {current_lr:.2e}")

        # Early stopping check
        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            print(f"   Best validation F1: {best_val_f1:.4f} at epoch {best_epoch}")
            print(f"   Best validation Loss: {best_val_loss:.4f} at epoch {best_epoch_loss}")
            break

    # Restore best model state
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nRestored model from epoch {best_epoch} with best {scheduler_metric}")

    print(f"Training completed!")
    print(f"Final Results:")
    print(f" • Best validation F1: {best_val_f1:.4f} (Epoch {best_epoch})")
    print(f" • Best validation Loss: {best_val_loss:.4f} (Epoch {best_epoch_loss})")
    print(f" • Total epochs trained: {len(val_f1_scores)}")
    print(f" • Early stopping: {'Yes' if epochs_without_improvement >= patience else 'No'}")
    print(f" • Final val loss at best F1: {val_losses[best_epoch-1] if best_epoch <= len(val_losses) else 'N/A':.4f}")
    if l1_lambda > 0 or l2_lambda > 0:
        print(f"Regularization applied - L1: {l1_lambda:.2e}, L2: {l2_lambda:.2e}")
    return train_losses, val_losses, val_f1_scores, train_f1_scores, best_epoch


### 7.2 Attention Pooling Module Definition

In [22]:

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.score = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, rnn_outputs, mask=None):
        # rnn_outputs: [B, T, H]
        scores = self.score(rnn_outputs).squeeze(-1)  # [B, T]
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = F.softmax(scores, dim=1)            # [B, T]
        context = torch.bmm(weights.unsqueeze(1), rnn_outputs)  # [B, 1, H]
        return context.squeeze(1), weights             # [B, H], [B, T]



### 7.3 Conv Block Definition

In [23]:
# ---------------------------
# Conv block for 1D time convs
# ---------------------------
class ConvBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=None, dropout=0.0):
        super().__init__()
        if padding is None:
            padding = (kernel_size - 1) // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding)
        self.bn = nn.BatchNorm1d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        # x: [B, C, T]
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.dropout(x)
        return x  # [B, out_ch, T]

### 7.4 Model Class Definition

In [24]:
class ConvRNNAttentionModel(nn.Module):
    """
    Conv-Enhanced RNN classifier with embedding layers for categorical features and attention.
    Combines embeddings with continuous features, applies 1D convolution for temporal
    feature extraction, passes to RNN for sequence modeling, and uses attention pooling.
    """
    def __init__(
            self,
            continuous_input_size,
            categorical_features,
            embedding_dims,
            hidden_size,
            num_layers,
            num_classes,
            rnn_type='LSTM',
            bidirectional=True,
            dropout_rate=0.2,
            conv_channels=[64, 32],
            conv_kernel_size=3,
            conv_dropout=0.2,
            use_conv=True,
            time_embedding_dim=0,
            max_time_value=0,
            use_attention=True
            ):
        super().__init__()

        self.rnn_type = rnn_type
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional
        self.categorical_features = list(categorical_features.keys())
        self.continuous_input_size = continuous_input_size
        self.use_conv = use_conv
        self.time_embedding_dim = time_embedding_dim
        self.use_attention = use_attention

        # Store embedding bounds for safe indexing
        self.embedding_num_embeddings = {}

        # Create embedding layers for each categorical feature
        self.embeddings = nn.ModuleDict()
        total_embedding_size = 0

        for feature, num_values in categorical_features.items():
            embed_dim = embedding_dims[feature]
            # Create embedding with num_values + 1 to handle 0 to num_values range
            num_embeddings = num_values + 1
            self.embeddings[feature] = nn.Embedding(num_embeddings, embed_dim)
            self.embedding_num_embeddings[feature] = num_embeddings
            total_embedding_size += embed_dim


        # Add time embedding layer
        self.time_embedding = nn.Embedding(max_time_value + 1, time_embedding_dim)
        total_embedding_size += time_embedding_dim

        # Total input size after embedding concatenation
        combined_input_size = continuous_input_size + total_embedding_size


        # Convolutional layers for temporal feature extraction
        if self.use_conv and len(conv_channels) > 0:
            conv_layers = []
            in_channels = combined_input_size

            for i, out_channels in enumerate(conv_channels):
                conv_layers.extend([
                    nn.Conv1d(in_channels, out_channels,
                             kernel_size=conv_kernel_size,
                             padding=conv_kernel_size//2),
                    nn.BatchNorm1d(out_channels),
                    nn.ReLU(),
                    nn.Dropout1d(conv_dropout)
                ])
                in_channels = out_channels

            self.conv_layers = nn.Sequential(*conv_layers)
            rnn_input_size = conv_channels[-1]
        else:
            self.conv_layers = None
            rnn_input_size = combined_input_size

        # Map string name to PyTorch RNN class
        rnn_map = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}

        if rnn_type not in rnn_map:
            raise ValueError("rnn_type must be 'RNN', 'LSTM', or 'GRU'")

        rnn_module = rnn_map[rnn_type]
        dropout_val = dropout_rate if num_layers > 1 else 0

        # Create the recurrent layer
        self.rnn = rnn_module(
            input_size=rnn_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout_val
        )

        # Calculate RNN output size
        rnn_output_size = hidden_size * 2 if bidirectional else hidden_size

        # NEW: Add attention pooling layer
        if self.use_attention:
            self.attention = AttentionPooling(rnn_output_size)
            classifier_input_size = rnn_output_size
            print(f"Attention pooling enabled (input size: {rnn_output_size})")
        else:
            classifier_input_size = rnn_output_size
            print(f"Attention pooling disabled. Using last hidden state.")

        # Final classification layers
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(classifier_input_size, classifier_input_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate / 2),
            nn.Linear(classifier_input_size // 2, num_classes)
        )

        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x, categorical_indices, time_indices=None):
        """
        x: (batch_size, seq_length, continuous_features)
        categorical_indices: Dict of {feature_name: (batch_size, seq_length)} tensors
        time_indices: (batch_size, seq_length) tensor with time values

        Returns:
        - output: (batch_size, num_classes) classification logits
        - attention_weights: (batch_size, seq_length) if attention is used, else None
        """
        batch_size, seq_length, _ = x.shape

        # Process embeddings for each categorical feature
        embedded_features = []

        for feature_name in self.categorical_features:
            if feature_name in categorical_indices:
                cat_indices = categorical_indices[feature_name]
                # CRITICAL FIX: Clamp indices to valid range [0, num_embeddings-1]
                max_idx = self.embedding_num_embeddings[feature_name] - 1
                cat_indices_clamped = torch.clamp(cat_indices.long(), 0, max_idx)
                embedded = self.embeddings[feature_name](cat_indices_clamped)
                embedded_features.append(embedded)

        # Process time embedding if provided
        if time_indices is not None:
            time_indices_clamped = torch.clamp(time_indices.long(), 0, self.time_embedding.num_embeddings - 1)
            time_embedded = self.time_embedding(time_indices_clamped)
            embedded_features.append(time_embedded)

        # Concatenate all embeddings with continuous features
        if embedded_features:
            all_embeddings = torch.cat(embedded_features, dim=-1)
            combined_input = torch.cat([x, all_embeddings], dim=-1)
        else:
            combined_input = x

        # Apply convolutional layers if enabled
        if self.use_conv and self.conv_layers is not None:
            conv_input = combined_input.transpose(1, 2)
            conv_output = self.conv_layers(conv_input)
            rnn_input = conv_output.transpose(1, 2)
        else:
            rnn_input = combined_input

        # Pass through RNN
        rnn_out, hidden = self.rnn(rnn_input)  # rnn_out: (batch, seq_len, hidden*directions)

        # NEW: Use attention pooling or last hidden state
        if self.use_attention:
            # Apply attention pooling over sequence
            context, attention_weights = self.attention(rnn_out)  # context: (batch, hidden*directions)
            classifier_input = context
        else:
            # Use last hidden state (original behavior)
            if self.rnn_type in ['LSTM', 'GRU']:
                if isinstance(hidden, tuple):
                    last_hidden = hidden[0]
                else:
                    last_hidden = hidden
            else:
                last_hidden = hidden

            if self.bidirectional:
                last_hidden = torch.cat([last_hidden[-2], last_hidden[-1]], dim=1)
            else:
                last_hidden = last_hidden[-1]

            classifier_input = last_hidden
            attention_weights = None

        # Apply classifier
        output = self.classifier(classifier_input)

        return output, attention_weights


### 7.5 Criterion (Focal Loss) Class Definition

In [25]:
# TO WEIGHT MORE THE "MORE DIFFICULT" CASES AND THE LESS FREQUENT LABELS:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=alpha, reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

## 8. Model Creation

### 8.1 Prepare for Embedding

In [26]:
# Define categorical feature parameters for the model
categorical_feature_config = {
    'pain_survey_1': 2,  # values: 0, 1, 2
    'pain_survey_2': 2,  # values: 0, 1, 2
    'pain_survey_3': 2,  # values: 0, 1, 2
    'pain_survey_4': 2,   # values: 0, 1, 2
    'has_peg_leg': 1,
    'has_hook_hand': 1,
    'has_eye_patch': 1
}

# Define embedding dimensions for each categorical feature
embedding_dims = {
    'pain_survey_1': 2,
    'pain_survey_2': 2,
    'pain_survey_3': 2,
    'pain_survey_4': 2,
    'has_peg_leg': 1,
    'has_hook_hand': 1,
    'has_eye_patch': 1
}

# Fix negative categorical values by shifting to 0-based indexing
# ( Basically shifts the range to start at 0)
for feature_name in ['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4']:
    if feature_name in X_train_cat:
        min_val = X_train_cat[feature_name].min()
        if min_val < 0:
            print(f"   • Shifting {feature_name} by {-min_val} (min was {min_val})")
            X_train_cat[feature_name] = X_train_cat[feature_name] - min_val
            X_val_cat[feature_name] = X_val_cat[feature_name] - min_val
            # Update config to reflect actual max value after shift
            new_max = X_train_cat[feature_name].max()
            categorical_feature_config[feature_name] = int(new_max)
            print(f"      New range: 0 to {new_max}")


### 8.2 Calculate the Input Sizes for Features

In [27]:
# Calculate correct input size
continuous_input_size = X_train_cont.shape[-1]  # Should be  30 (we drop joint_30)
total_embedding_size = sum(embedding_dims.values())  # 2+2+2+2+1+1+1 = 11
combined_input_size = continuous_input_size + total_embedding_size  # 30(cont) + 11(emb) = 41

print(f"\n🔍 Input size calculation:")
print(f"   • Continuous features: {continuous_input_size}")
print(f"   • Total embeddings: {total_embedding_size}")
print(f"   • Combined input: {combined_input_size}")




🔍 Input size calculation:
   • Continuous features: 30
   • Total embeddings: 11
   • Combined input: 41


In [28]:




#CRITERION: Focal Loss to handle class imbalance
alpha = None
# Gamma: higher = more focus on hard examples (try 2.3, 3.0, or 3.5)
criterion = FocalLoss(alpha=alpha, gamma=2.3)

## 9. Prepare the Submission Dataset

In [29]:
# Load the actual test dataset (this doesn't have labels)
X_test_final_df = pd.read_csv('an2dl2526c1/pirate_pain_test.csv')

df_corr_check = X_test_final_df.copy()
df_corr_check['has_peg_leg'] = np.where(df_corr_check['n_legs'] == 'one+peg_leg', 1, 0)
df_corr_check['has_hook_hand'] = np.where(df_corr_check['n_hands'] == 'one+hook_hand', 1, 0)
df_corr_check['has_eye_patch'] = np.where(df_corr_check['n_eyes'] == 'one+eye_patch', 1, 0)

X_test_final_df = df_corr_check.copy()

In [30]:
def build_sequences_test(df, window=200, stride=200):
    assert window % stride == 0

    dataset = []

    # Get feature columns (exclude sample_index and time)
    columns = [col for col in df.columns if col not in ['sample_index', 'time']]

    for id in df['sample_index'].unique():
        temp = df[df['sample_index'] == id][columns].values

        # Padding
        padding_len = (window - len(temp) % window) % window
        padding = np.zeros((padding_len, len(columns)), dtype='float32')
        temp = np.concatenate((temp, padding))

        # Build windows
        idx = 0
        while idx + window <= len(temp):
            dataset.append(temp[idx:idx + window])
            idx += stride

    return np.array(dataset)

In [31]:
from sklearn.preprocessing import StandardScaler

# ============================================================
# STANDARDIZE TEST DATA (USING TRAINING SCALER)
# ============================================================
print("=" * 60)
print("TEST DATA STANDARDIZATION")
print("=" * 60)

# Drop n_legs, n_hands, n_eyes first (before scaling)
X_test_final_df = X_test_final_df.drop(columns=['n_legs', 'n_hands', 'n_eyes', 'joint_30'])

# Define columns to scale (must match training data columns)
scale_columns = [
    col for col in data.columns
    if (col.startswith('joint_') )
]


print(f"\nFeatures to standardize: {len(scale_columns)}")
print(f"   • Continuous features: {[col for col in scale_columns if col.startswith('joint')]}")
print(f"   • Categorical features: {[col for col in scale_columns if col.startswith('pain_survey')]}")

# Check for NaN values before scaling
print(f"\n Checking for missing values in test data...")
test_nans = X_test_final_df[scale_columns].isna().sum().sum()

if test_nans > 0:
    print(f"     Warning: Found {test_nans} NaN values!")
    print(f"   • Filling NaNs with column mean from test data...")
    test_means = X_test_final_df[scale_columns].mean()
    X_test_final_df[scale_columns] = X_test_final_df[scale_columns].fillna(test_means)
else:
    print(f"   ✅ No missing values found")

#  IMPORTANT: Use the SAME scaler that was fitted on training data
# If you don't have the training scaler saved, you need to use it from the training section
if 'scaler' not in globals():
    print("\n  WARNING: Training scaler not found in globals!")
    print("   Creating NEW scaler fitted on test data (NOT RECOMMENDED for production)")
    print("   For best results, use the scaler fitted on training data.")
    scaler = StandardScaler()
    scaler.fit(X_test_final_df[scale_columns])
else:
    print("\n Using scaler from training data")

# Transform test data using the scaler
print(f"\n Standardizing test data...")
X_test_final_df[scale_columns] = scaler.transform(X_test_final_df[scale_columns])

# Verify standardization results
print(f"\n Standardization completed!")
print(f"   • Test set mean: {X_test_final_df[scale_columns].mean().mean():.6f}")
print(f"   • Test set std: {X_test_final_df[scale_columns].std().mean():.6f}")
print(f"   • Test set min/max: [{X_test_final_df[scale_columns].min().min():.4f}, {X_test_final_df[scale_columns].max().max():.4f}]")

# Show sample of scaled data
print(f"\n Sample of standardized test data:")
print(X_test_final_df.head())

print("\n" + "=" * 60)
print(" Test data standardization completed!")
print("=" * 60)

TEST DATA STANDARDIZATION

Features to standardize: 30
   • Continuous features: ['joint_00', 'joint_01', 'joint_02', 'joint_03', 'joint_04', 'joint_05', 'joint_06', 'joint_07', 'joint_08', 'joint_09', 'joint_10', 'joint_11', 'joint_12', 'joint_13', 'joint_14', 'joint_15', 'joint_16', 'joint_17', 'joint_18', 'joint_19', 'joint_20', 'joint_21', 'joint_22', 'joint_23', 'joint_24', 'joint_25', 'joint_26', 'joint_27', 'joint_28', 'joint_29']
   • Categorical features: []

 Checking for missing values in test data...
   ✅ No missing values found

 Using scaler from training data

 Standardizing test data...

 Standardization completed!
   • Test set mean: 0.100769
   • Test set std: 1.059345
   • Test set min/max: [-7.5996, 377.9331]

 Sample of standardized test data:
   sample_index  time  pain_survey_1  pain_survey_2  pain_survey_3  \
0             0     0              2              2              2   
1             0     1              2              2              2   
2             0

In [32]:
# Build sequences from the actual test data for Enhanced Model
print(f"Building ENHANCED sequences for actual test dataset with WINDOW_SIZE={WINDOW_SIZE}, STRIDE={STRIDE}")

# Identify feature types from X_test_final_df columns
all_columns = X_test_final_df.columns.tolist()
print(f"Available columns: {all_columns}")

# Define feature separation - MUST match training data!
categorical_features = ['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4',
                         'has_peg_leg', 'has_hook_hand', 'has_eye_patch']
joint_features = [col for col in all_columns if col.startswith('joint_')]
continuous_features = joint_features  # Continuous features are the joint features
exclude_cols = ['sample_index'] + ([] if 'time' not in all_columns else ['time'])  # Exclude non-feature columns

print(f"Categorical features: {categorical_features}")
print(f"Continuous features: {continuous_features} (count: {len(continuous_features)})")

# Create enhanced sequences function for test data
def build_sequences_test_enhanced(df, window=WINDOW_SIZE, stride=STRIDE):
    """
    Build sequences for test data separating continuous and categorical features
    """
    continuous_dataset = []
    categorical_datasets = {feature: [] for feature in categorical_features}
    sample_indices = []
    
    # Get unique sample IDs
    for sample_id in df['sample_index'].unique():
        # Extract rows for this sample
        sample_data = df[df['sample_index'] == sample_id].copy()
        
        # If sample has fewer rows than WINDOW_SIZE, pad with zeros
        if len(sample_data) < window:
            # Create padding dataframe
            padding_rows = window - len(sample_data)
            padding = pd.DataFrame(0, index=range(padding_rows), columns=sample_data.columns)
            sample_data = pd.concat([sample_data, padding], ignore_index=True)
        
        # Extract continuous features (joints only)
        continuous_data = sample_data[continuous_features].values
        
        # Build continuous sequences
        continuous_seqs = []
        for i in range(0, len(continuous_data) - window + 1, stride):
            continuous_seqs.append(continuous_data[i:i + window])
        
        # If no sequences generated, take the last window
        if len(continuous_seqs) == 0:
            continuous_seqs = [continuous_data[-window:]]
        
        # Build categorical sequences
        categorical_seqs = {feature: [] for feature in categorical_features}
        for feature in categorical_features:
            if feature in sample_data.columns:
                cat_data = sample_data[feature].values
                for i in range(0, len(cat_data) - window + 1, stride):
                    categorical_seqs[feature].append(cat_data[i:i + window])
                # If no sequences generated, take the last window
                if len(categorical_seqs[feature]) == 0:
                    categorical_seqs[feature] = [cat_data[-window:]]
            else:
                # If feature doesn't exist, create zero sequences
                print(f"⚠️  Warning: {feature} not found in data, using zeros")
                for _ in continuous_seqs:
                    categorical_seqs[feature].append(np.zeros(window, dtype='int64'))
        
        # Store sequences (take first sequence for each sample)
        continuous_dataset.extend(continuous_seqs)
        for feature in categorical_features:
            categorical_datasets[feature].extend(categorical_seqs[feature])
        sample_indices.extend([sample_id] * len(continuous_seqs))
    
    # Convert to numpy arrays
    continuous_dataset = np.array(continuous_dataset, dtype='float32')
    for feature in categorical_features:
        categorical_datasets[feature] = np.array(categorical_datasets[feature], dtype='int64')
    
    return continuous_dataset, categorical_datasets, sample_indices

# Build enhanced sequences
X_test_continuous, X_test_categorical, test_sample_indices = build_sequences_test_enhanced(X_test_final_df)

# Fix categorical values to match training (shift negative values)
print("\n🔧 Fixing categorical feature values in test data...")
for feature_name in ['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4']:
    if feature_name in X_test_categorical:
        min_val = X_test_categorical[feature_name].min()
        if min_val < 0:
            print(f"   • Shifting {feature_name} by {-min_val} (min was {min_val})")
            X_test_categorical[feature_name] = X_test_categorical[feature_name] - min_val

# Handle NaN values
if np.isnan(X_test_continuous).any():
    X_test_continuous = np.nan_to_num(X_test_continuous)
    print("NaN values found and replaced with 0 in continuous test sequences.")

for feature in categorical_features:
    if np.isnan(X_test_categorical[feature]).any():
        X_test_categorical[feature] = np.nan_to_num(X_test_categorical[feature]).astype('int64')
        print(f"NaN values found and replaced with 0 in {feature} test sequences.")

print(f"\nEnhanced test sequences shapes:")
print(f"  Continuous: {X_test_continuous.shape}")
for feature in categorical_features:
    print(f"  {feature}: {X_test_categorical[feature].shape}")
print(f"  Sample indices: {len(test_sample_indices)}")

# Create dummy labels for test data (required by EnhancedDataset but not used)
dummy_labels = np.zeros(len(test_sample_indices), dtype='int64')

# Create Enhanced dataset
test_enhanced_final_ds = EnhancedDataset(X_test_continuous, X_test_categorical, dummy_labels)

# Create Enhanced DataLoader
test_enhanced_final_loader = make_enhanced_loader(
    test_enhanced_final_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    sampler=None
)

print(f"\n✅ Enhanced test dataset created successfully!")
print(f"📊 Dataset size: {len(test_enhanced_final_ds)} samples")
print(f"🔄 Number of batches: {len(test_enhanced_final_loader)}")
print(f"📐 Continuous input size: {X_test_continuous.shape[-1]}")
print(f"🏷️ Categorical features: {list(X_test_categorical.keys())}" )

# Test the enhanced loader
print(f"\n🧪 Testing Enhanced DataLoader...")
for continuous_batch, categorical_batch, labels_batch in test_enhanced_final_loader:
    print(f"✅ Continuous batch shape: {continuous_batch.shape}")
    print(f"✅ Labels batch shape: {labels_batch.shape}")
    for feature, data in categorical_batch.items():
        print(f"✅ {feature} batch shape: {data.shape}")
    break

print(f"\n🎯 Enhanced test data is ready for the Enhanced model!")

Building ENHANCED sequences for actual test dataset with WINDOW_SIZE=10, STRIDE=2
Available columns: ['sample_index', 'time', 'pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4', 'joint_00', 'joint_01', 'joint_02', 'joint_03', 'joint_04', 'joint_05', 'joint_06', 'joint_07', 'joint_08', 'joint_09', 'joint_10', 'joint_11', 'joint_12', 'joint_13', 'joint_14', 'joint_15', 'joint_16', 'joint_17', 'joint_18', 'joint_19', 'joint_20', 'joint_21', 'joint_22', 'joint_23', 'joint_24', 'joint_25', 'joint_26', 'joint_27', 'joint_28', 'joint_29', 'has_peg_leg', 'has_hook_hand', 'has_eye_patch']
Categorical features: ['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4', 'has_peg_leg', 'has_hook_hand', 'has_eye_patch']
Continuous features: ['joint_00', 'joint_01', 'joint_02', 'joint_03', 'joint_04', 'joint_05', 'joint_06', 'joint_07', 'joint_08', 'joint_09', 'joint_10', 'joint_11', 'joint_12', 'joint_13', 'joint_14', 'joint_15', 'joint_16', 'joint_17', 'joint_18', 'joint_

## 10. Grid Search  

In [33]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import json
import os
import torch
from torch.utils.data import DataLoader, Subset, TensorDataset
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold, train_test_split
import numpy as np
import random
import time

############################################################
# 10.1 Shared utilities
############################################################


os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

############################################################
# 10.2 Build a unified TensorDataset compatible with the model
############################################################

# We store: (continuous, categorical_dict, label)
# For Optuna we will pass these into EnhancedDataset inside each trial.

all_cont = np.concatenate([X_train_cont, X_val_cont, X_test_cont], axis=0)
all_labels = np.concatenate([y_train, y_val, y_test_enh], axis=0)

# For categorical, all feature arrays have shape (N_seq, seq_len)
all_cat = {}
for feat in X_train_cat.keys():
    all_cat[feat] = np.concatenate([
        X_train_cat[feat],
        X_val_cat[feat],
        X_test_cat[feat]
    ], axis=0)

# We'll store indices [0..N-1] here and slice numpy arrays inside the trial.
num_sequences = all_cont.shape[0]
all_indices = np.arange(num_sequences)

############################################################
# 10.3 Define hyperparameter search space based on Section 1
############################################################

# Section 1 globals:
#   RNN_TYPE, BIDIRECTIONAL, BATCH_SIZE, LEARNING_RATE, EPOCHS,
#   PATIENCE, HIDDEN_LAYERS, HIDDEN_SIZE, DROPOUT_RATE,
#   L1_LAMBDA, L2_LAMBDA, CONV_CHANNELS, CONV_KERNEL_SIZES,
#   CONV_DROPOUT_RATE, WINDOW_SIZE, STRIDE, NUM_CLASS

def sample_hparams(trial: optuna.trial.Trial):
    """Sample hyperparameters around the base values from Section 1."""

    # Hidden size: around base HIDDEN_SIZE
    hidden_choices = sorted({
        max(16, HIDDEN_SIZE // 2),
        HIDDEN_SIZE,
        HIDDEN_SIZE * 2
    })

    # Hidden layers: 1..4 but include current HIDDEN_LAYERS
    num_layers_choices = sorted({1, 2, 3, 4, int(HIDDEN_LAYERS)})

    # Dropout: around base DROPOUT_RATE
    dr = DROPOUT_RATE
    dropout_choices = sorted({
        round(max(0.0, dr - 0.2), 2),
        round(dr, 2),
        round(min(0.7, dr + 0.2), 2)
    })

    # RNN type: allow current RNN_TYPE and {LSTM, GRU}
    rnn_choices = sorted(set([RNN_TYPE, "LSTM", "GRU"]))

    # Bidirectional: keep current value, optionally also False
    bi_choices = sorted(set([BIDIRECTIONAL, True, False]))

    # Learning rate: search around LEARNING_RATE on log scale
    lr_low = LEARNING_RATE / 5.0
    lr_high = LEARNING_RATE * 5.0

    # Batch size: allow base BATCH_SIZE and some neighbors
    bs_choices = sorted({max(16, BATCH_SIZE // 2), BATCH_SIZE, BATCH_SIZE * 2})

    # L1 & L2 lambdas: use log scale around globals
    l1_base = max(1e-9, L1_LAMBDA)
    l2_base = max(1e-9, L2_LAMBDA)

    # Conv channels: allow a few variants around CONV_CHANNELS
    base_channels = CONV_CHANNELS
    # simple variants: original, doubled, single-channel
    conv_channel_options = [
        base_channels,
        [c * 2 for c in base_channels],
        [base_channels[0]]
    ]

    # Conv kernel size: pick from small integers, include existing
    conv_kernel_choices = sorted(set([3, 5, 7] + CONV_KERNEL_SIZES))

    # Conv dropout around base
    cd = CONV_DROPOUT_RATE
    conv_dropout_choices = sorted({
        round(max(0.0, cd - 0.2), 2),
        round(cd, 2),
        round(min(0.7, cd + 0.2), 2)
    })

    hparams = {
        "hidden_size": trial.suggest_categorical("hidden_size", hidden_choices),
        "num_layers": trial.suggest_categorical("num_layers", num_layers_choices),
        "dropout_rate": trial.suggest_categorical("dropout_rate", dropout_choices),
        "rnn_type": trial.suggest_categorical("rnn_type", rnn_choices),
        "bidirectional": trial.suggest_categorical("bidirectional", bi_choices),
        "learning_rate": trial.suggest_float("learning_rate", lr_low, lr_high, log=True),
        "batch_size": trial.suggest_categorical("batch_size", bs_choices),
        "l1_lambda": trial.suggest_float("l1_lambda", l1_base / 10.0, l1_base * 10.0, log=True),
        "l2_lambda": trial.suggest_float("l2_lambda", l2_base / 10.0, l2_base * 10.0, log=True),
        "conv_channels": trial.suggest_categorical("conv_channels", conv_channel_options),
        "conv_kernel_size": trial.suggest_categorical("conv_kernel_size", conv_kernel_choices),
        "conv_dropout": trial.suggest_categorical("conv_dropout", conv_dropout_choices)
    }

    return hparams

############################################################
# 10.4 Utility: keep top models
############################################################

def save_top_models(model_state, f1, trial_number):
    path = os.path.join(SAVE_DIR, f"trial{trial_number}_f1_{f1:.4f}.pt")
    torch.save(model_state, path)
    saved = [f for f in os.listdir(SAVE_DIR) if f.endswith(".pt")]

    if len(saved) > 20:
        scored = []
        for f in saved:
            try:
                s = float(f.split("_f1_")[1].replace(".pt", ""))
                scored.append((s, f))
            except Exception:
                pass
        scored.sort(reverse=True, key=lambda x: x[0])
        for _, fname in scored[20:]:
            os.remove(os.path.join(SAVE_DIR, fname))

############################################################
# 10.5 Single training run wrapper for Optuna, using
#      train_enhanced_model and ConvRNNAttentionModel
############################################################

def make_enhanced_dataset(indices):
    """Build EnhancedDataset from global numpy arrays using the given indices."""
    cont = all_cont[indices]
    labels = all_labels[indices]
    cat_dict = {feat: all_cat[feat][indices] for feat in all_cat.keys()}
    # pirate_ids are not needed for grid search, pass None
    ds = EnhancedDataset(cont, cat_dict, labels, pirate_ids=None)
    return ds

def train_one_run_enhanced(hparams, train_idx, val_idx, trial=None):
    # Build datasets & loaders
    train_ds = make_enhanced_dataset(train_idx)
    val_ds = make_enhanced_dataset(val_idx)

    train_loader = make_enhanced_loader(
        train_ds,
        batch_size=hparams["batch_size"],
        shuffle=True,
        drop_last=False,
        sampler=None,
    )

    val_loader = make_enhanced_loader(
        val_ds,
        batch_size=hparams["batch_size"],
        shuffle=False,
        drop_last=False,
        sampler=None,
    )

    # Model creation: same architecture class as section 8,
    # but hyperparameters come from hparams.
    model = ConvRNNAttentionModel(
        continuous_input_size=X_train_cont.shape[-1],
        categorical_features=categorical_feature_config,
        embedding_dims=embedding_dims,
        hidden_size=hparams["hidden_size"],
        num_layers=hparams["num_layers"],
        num_classes=NUM_CLASS,
        rnn_type=hparams["rnn_type"],
        bidirectional=hparams["bidirectional"],
        dropout_rate=hparams["dropout_rate"],
        conv_channels=hparams["conv_channels"],
        conv_kernel_size=hparams["conv_kernel_size"],
        conv_dropout=hparams["conv_dropout"],
        use_conv=True,
        time_embedding_dim=0,
        max_time_value=0,
        use_attention=True,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=hparams["learning_rate"],
        weight_decay=hparams["l2_lambda"],
    )

    # Use FocalLoss as before
    local_criterion = criterion  # already defined in section 8

    # We will cap epochs per trial and use smaller patience
    train_losses, val_losses, val_f1s, train_f1s, best_epoch = train_enhanced_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=local_criterion,
        optimizer=optimizer,
        scheduler=None,
        epochs=MAX_EPOCHS,
        l1_lambda=hparams["l1_lambda"],
        l2_lambda=0.0,  # L2 via optimizer weight_decay
        patience=GRID_PATIENCE,
        min_delta=0.001,
        scheduler_metric='f1',
    )

    best_f1 = max(val_f1s) if len(val_f1s) > 0 else 0.0

    if VERBOSE_GRID:
        print(f"   -> Fold best F1: {best_f1:.4f} (epoch {best_epoch})")

    if trial is not None:
        trial.report(best_f1, step=best_epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    # Return final model (best weights already restored in train_enhanced_model) and F1
    return model, best_f1

############################################################
# 10.6 Optuna objective with optional K-Fold cross-validation
############################################################

def objective(trial: optuna.trial.Trial):
    start_time = time.time()

    set_seed(GRID_SEED)
    hparams = sample_hparams(trial)

    print("\n====================================================")
    print(f">>> Trial {trial.number} START")
    print("Hyperparameters:")
    for k, v in hparams.items():
        print(f"  {k}: {v}")
    print("====================================================\n")

    fold_scores = []
    best_fold_f1 = -1.0
    best_state_dict = None

    if USE_KFOLD:
        # K-fold on index array
        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=GRID_SEED)

        for fold, (train_idx, val_idx) in enumerate(kf.split(all_indices)):
            print(f"----- Fold {fold+1}/{N_FOLDS} -----")

            model, f1_fold = train_one_run_enhanced(hparams, train_idx, val_idx, trial)
            fold_scores.append(f1_fold)

            if f1_fold > best_fold_f1:
                best_fold_f1 = f1_fold
                best_state_dict = model.state_dict()

            print(f"Fold {fold+1} F1 = {f1_fold:.4f}")
    else:
        # Single train/validation split on indices
        train_idx, val_idx = train_test_split(
            all_indices,
            test_size=SINGLE_VAL_FRACTION,
            stratify=all_labels,
            random_state=GRID_SEED,
        )
        print(f"Using single train/val split with val fraction={SINGLE_VAL_FRACTION}")
        model, f1_single = train_one_run_enhanced(hparams, train_idx, val_idx, trial)
        fold_scores.append(f1_single)
        best_fold_f1 = f1_single
        best_state_dict = model.state_dict()
        print(f"Single-split F1 = {f1_single:.4f}")

    mean_f1 = float(np.mean(fold_scores)) if len(fold_scores) > 0 else 0.0
    end_time = time.time()

    print(f"\n>>> Trial {trial.number} END — Mean F1 = {mean_f1:.4f}")
    print(f"Duration: {end_time - start_time:.2f} seconds")
    print("====================================================\n")

    # Save best model for this trial
    if best_state_dict is not None:
        save_top_models(best_state_dict, mean_f1, trial.number)

        # === PER-TRIAL SUBMISSION GENERATION ===
        # Rebuild model with this trial's hyperparameters
        model_for_sub = ConvRNNAttentionModel(
            continuous_input_size=X_train_cont.shape[-1],
            categorical_features=categorical_feature_config,
            embedding_dims=embedding_dims,
            hidden_size=hparams["hidden_size"],
            num_layers=hparams["num_layers"],
            num_classes=NUM_CLASS,
            rnn_type=hparams["rnn_type"],
            bidirectional=hparams["bidirectional"],
            dropout_rate=hparams["dropout_rate"],
            conv_channels=hparams["conv_channels"],
            conv_kernel_size=hparams["conv_kernel_size"],
            conv_dropout=hparams["conv_dropout"],
            use_conv=True,
            time_embedding_dim=0,
            max_time_value=0,
            use_attention=True,
        ).to(device)

        model_for_sub.load_state_dict(best_state_dict)
        model_for_sub.eval()

        from collections import Counter
        label_mapping = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}

        all_predictions = []
        all_sample_indices = []
        sample_ptr = 0

        with torch.no_grad():
            for batch in test_enhanced_final_loader:
                continuous_batch, categorical_batch, labels_batch = batch
                continuous_batch = continuous_batch.to(device)
                categorical_batch = {k: v.to(device) for k, v in categorical_batch.items()}

                batch_size = continuous_batch.shape[0]
                batch_sample_indices = test_sample_indices[sample_ptr:sample_ptr + batch_size]
                sample_ptr += batch_size

                outputs, _ = model_for_sub(continuous_batch, categorical_batch)
                _, predicted = torch.max(outputs, 1)

                all_predictions.extend(predicted.cpu().numpy())
                all_sample_indices.extend(batch_sample_indices)

        sample_predictions = {}
        for sample_id, pred in zip(all_sample_indices, all_predictions):
            sample_id_int = int(float(sample_id))
            if sample_id_int not in sample_predictions:
                sample_predictions[sample_id_int] = []
            sample_predictions[sample_id_int].append(int(pred))

        final_predictions = {}
        for sample_id, preds in sample_predictions.items():
            most_common = Counter(preds).most_common(1)[0][0]
            final_predictions[sample_id] = most_common

        final_labels = {sid: label_mapping[pred] for sid, pred in final_predictions.items()}
        submission_rows = []
        for sid in sorted(final_predictions.keys()):
            submission_rows.append({
                'sample_index': f"{int(sid):03d}",
                'label': final_labels[sid]
            })
        submission_df = pd.DataFrame(submission_rows)

        os.makedirs('submissions', exist_ok=True)
        submission_filename = f"trial{trial.number}_F1_{mean_f1:.4f}.csv"
        submission_path = os.path.join('submissions', submission_filename)
        submission_df.to_csv(submission_path, index=False)
        print(f"✅ Saved submission for trial {trial.number} to {submission_path}")

    # Save hyperparams for this trial
    with open(os.path.join(SAVE_DIR, f"trial{trial.number}_f1_{mean_f1:.4f}_hparams.json"), "w") as f:
        json.dump(hparams, f, indent=4)

    return mean_f1

############################################################
# 10.7 Run Optuna study
############################################################

study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=GRID_SEED, multivariate=True),
    pruner=MedianPruner(n_warmup_steps=3),
)

print("Starting Optuna Hyperparameter Search over Section 1 params…")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\n===========================================")
print("Search Complete!")
print(f"Best F1: {study.best_value:.4f}")
print("Best Params:")
for k, v in study.best_trial.params.items():
    print(f"  {k}: {v}")
print("Models saved in:", SAVE_DIR)
print("===========================================")


[I 2025-11-17 21:34:42,578] A new study created in memory with name: no-name-eb993304-f098-4826-a02b-4ea03a9998c8


Starting Optuna Hyperparameter Search over Section 1 params…


  0%|          | 0/2 [00:00<?, ?it/s]


>>> Trial 0 START
Hyperparameters:
  hidden_size: 64
  num_layers: 1
  dropout_rate: 0.0
  rnn_type: LSTM
  bidirectional: False
  learning_rate: 0.00035909585479487006
  batch_size: 1024
  l1_lambda: 7.309539835912894e-08
  l2_lambda: 0.0038234752246751854
  conv_channels: [64]
  conv_kernel_size: 7
  conv_dropout: 0.7

Using single train/val split with val fraction=0.2
Attention pooling enabled (input size: 64)
Starting enhanced model training with Early Stopping...
 Regularization: L1=7.31e-08, L2=0.00e+00
 Early Stopping: Patience=10 epochs, Min improvement=0.0010
 Scheduler Metric: f1
Starting enhanced model training with Early Stopping...
 Regularization: L1=7.31e-08, L2=0.00e+00
 Early Stopping: Patience=10 epochs, Min improvement=0.0010
 Scheduler Metric: f1
Epoch 1/60: Train Loss: 0.3228| Val Loss: 0.2319| Train F1: 0.6705| Val F1: 0.6742
Epoch 1/60: Train Loss: 0.3228| Val Loss: 0.2319| Train F1: 0.6705| Val F1: 0.6742
Epoch 2/60: Train Loss: 0.2191| Val Loss: 0.1852| Train 

Best trial: 0. Best value: 0.907473:  50%|█████     | 1/2 [00:59<00:59, 59.27s/it]

✅ Saved submission for trial 0 to submissions\trial0_F1_0.9075.csv
[I 2025-11-17 21:35:41,850] Trial 0 finished with value: 0.9074728760149867 and parameters: {'hidden_size': 64, 'num_layers': 1, 'dropout_rate': 0.0, 'rnn_type': 'LSTM', 'bidirectional': False, 'learning_rate': 0.00035909585479487006, 'batch_size': 1024, 'l1_lambda': 7.309539835912894e-08, 'l2_lambda': 0.0038234752246751854, 'conv_channels': [64], 'conv_kernel_size': 7, 'conv_dropout': 0.7}. Best is trial 0 with value: 0.9074728760149867.

>>> Trial 1 START
Hyperparameters:
  hidden_size: 64
  num_layers: 3
  dropout_rate: 0.4
  rnn_type: GRU
  bidirectional: False
  learning_rate: 0.003734266997106625
  batch_size: 512
  l1_lambda: 1.0968217207529512e-07
  l2_lambda: 0.0123999678368461
  conv_channels: [128]
  conv_kernel_size: 3
  conv_dropout: 0.4

Using single train/val split with val fraction=0.2
Attention pooling enabled (input size: 64)
Starting enhanced model training with Early Stopping...
 Regularization: L1=1

Best trial: 1. Best value: 0.991565: 100%|██████████| 2/2 [02:54<00:00, 87.15s/it]

✅ Saved submission for trial 1 to submissions\trial1_F1_0.9916.csv
[I 2025-11-17 21:37:36,885] Trial 1 finished with value: 0.9915649172212736 and parameters: {'hidden_size': 64, 'num_layers': 3, 'dropout_rate': 0.4, 'rnn_type': 'GRU', 'bidirectional': False, 'learning_rate': 0.003734266997106625, 'batch_size': 512, 'l1_lambda': 1.0968217207529512e-07, 'l2_lambda': 0.0123999678368461, 'conv_channels': [128], 'conv_kernel_size': 3, 'conv_dropout': 0.4}. Best is trial 1 with value: 0.9915649172212736.

Search Complete!
Best F1: 0.9916
Best Params:
  hidden_size: 64
  num_layers: 3
  dropout_rate: 0.4
  rnn_type: GRU
  bidirectional: False
  learning_rate: 0.003734266997106625
  batch_size: 512
  l1_lambda: 1.0968217207529512e-07
  l2_lambda: 0.0123999678368461
  conv_channels: [128]
  conv_kernel_size: 3
  conv_dropout: 0.4
Models saved in: optuna_top20
